# M7 Run 2 - honest OOF (folds 1-8) @5000 + ensemble preview = gate C1

First number **comparable to M3 (0.619) / M4 (0.718) / vote (0.736)** — pooled OOF AP over folds 1-8.
Config **identical to Run 1 baseline**; resolution fixed to **5000**.

For each held-out fold f in 1..8: train on the other 7 folds (all WPW + 12000 subsampled negatives), seeded stratified internal val for early-stop, 5-seed ensemble; predict fold f at **true prevalence**. **fold9** is predicted by reusing every OOF model (none saw fold9) = validation, looked at once. **fold10 NEVER touched.**

> Ends on **gate C1** (OOF >= 0.65 AND max rho(M7,{M3,M4}) < 0.50): unlocks/locks the pretraining block (Runs 8-10). rho<0.5 is necessary-not-sufficient; overriding a FAIL must be journaled.

> Prereq: PyTorch (CPU) + the M3/M4 OOF CSVs in data/processed (present).

### Block 2.0: Setup and configuration

In [ ]:
# M7 Run 2 - honest OOF (folds 1-8) @5000 + read-only ensemble preview = gate C1.
# Resolution fixed to 5000 (Run 1). Config IDENTICAL to Run 1 baseline (focal+balanced, 5 seeds, same aug).
# fold9 = validation (looked at once, via reuse of the OOF models). fold10 NEVER touched.
import os, sys, json, time, warnings
import numpy as np, pandas as pd
from scipy.signal import butter, sosfiltfilt
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score, roc_auc_score
from joblib import Parallel, delayed
from tqdm import tqdm
warnings.filterwarnings('ignore')
ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); SRC=os.path.join(ROOT,'src')
FIG=os.path.join(ROOT,'reports','figures'); METRICS=os.path.join(ROOT,'reports','metrics')
MODELS=os.path.join(ROOT,'models','M7_run2'); CACHE_DIR=os.path.join(PROCESSED,'m7_run2_cache')
for d in (FIG,METRICS,MODELS,CACHE_DIR): os.makedirs(d,exist_ok=True)
sys.path.insert(0,SRC)
from signal_loading import load_signal, LEADS_CANONICAL
from evaluation import evaluate_standard, _boot_ci
with open(os.path.join(PROCESSED,'filter_config.json'),encoding='utf-8') as f:
    FCFG=json.load(f)['filter_FINAL']
FS=FCFG['fs']; LEADS=list(LEADS_CANONICAL); LEAD_IDX={L:LEADS_CANONICAL.index(L) for L in LEADS}
SOS=butter(FCFG['order'],[FCFG['low']/(FS/2),FCFG['high']/(FS/2)],btype='band',output='sos')
def bp(x): return sosfiltfilt(SOS,np.asarray(x,dtype=np.float64))
# ---- config (single source of truth) ----
RANDOM_STATE=42
RESO=5000                  # native resolution
N_NEG_TRAIN=12000          # subsampled non-WPW pool per fold-model (balanced batches draw from it)
SEEDS=[0,1,2,3,4]
EPOCHS=60; PATIENCE=10; BATCH=64; LR=1e-3; WD=1e-4
FOCAL_GAMMA=2.0; FOCAL_ALPHA=0.5
VAL_FRAC=0.15              # internal early-stop split carved from the training pool (seeded, stratified)
N_JOBS=6
REF={'M3':0.619,'M4':0.718,'vote_M3_M4':0.736}     # OOF AP references
C1_OOF=0.65; C1_RHO=0.50                            # pre-registered gate thresholds
meta=pd.read_csv(os.path.join(PROCESSED,'metadata_combined.csv'),dtype={'ecg_id':str})
print('M7 Run 2 | OOF folds 1-8 @%d | seeds %s | gate C1 (OOF>=%.2f AND rho<%.2f)'%(RESO,SEEDS,C1_OOF,C1_RHO))
print('WPW folds1-8 %d | fold9 %d | fold10 %d [UNTOUCHED]'%(
    int(meta[(meta.fold.between(1,8))&(meta.label==1)].shape[0]),
    int(meta[(meta.fold==9)&(meta.label==1)].shape[0]),
    int(meta[(meta.fold==10)&(meta.label==1)].shape[0])))
print('Block 2.0 OK.')


### Block 2.1: Cache folds 1-8 + fold 9 (native 5000, z-scored, float16)

In [ ]:
# Cache folds 1-8 (OOF/train) + fold 9 (validation) at native 5000, z-scored, float16 (memory).
# Robust load (skip malformed headers, keep alignment). Carry ecg_id + source + fold + label for id-aligned preview.
def _build_rows(df):
    return list(zip(df.ecg_id.tolist(),df.source.tolist(),df.fold.tolist(),df.label.values.astype('float32').tolist()))
def _zscore(x):  # (12,5000) -> float16 z-scored per lead
    return ((x-x.mean(1,keepdims=True))/(x.std(1,keepdims=True)+1e-6)).astype('float16')
def _extract_chunk(rows, outpath):
    xs=[]; eids=[]; srcs=[]; folds=[]; ys=[]; failed=0
    for eid,src,fold,lab in rows:
        try:
            raw=load_signal(eid,src)
            x=np.stack([bp(raw[:,LEAD_IDX[Ld]]) for Ld in LEADS])   # (12,5000)
            xs.append(_zscore(np.nan_to_num(x))); eids.append(eid); srcs.append(str(src)); folds.append(int(fold)); ys.append(lab)
        except Exception:
            failed+=1
    arr=np.stack(xs).astype('float16') if xs else np.empty((0,12,5000),dtype='float16')
    np.save(outpath,arr)
    return outpath, eids, srcs, folds, ys, failed
def _extract(df, tag):
    rows=_build_rows(df)
    parts=[p for p in np.array_split(np.arange(len(rows)),N_JOBS) if len(p)]
    paths=[os.path.join(CACHE_DIR,'%s_chunk%d.npy'%(tag,j)) for j in range(len(parts))]
    res=Parallel(n_jobs=N_JOBS)(delayed(_extract_chunk)([rows[i] for i in part],p) for part,p in zip(parts,paths))
    X=np.concatenate([np.load(pth) for pth,_,_,_,_,_ in res],axis=0)
    eids=[e for _,es,_,_,_,_ in res for e in es]; srcs=[s for _,_,ss,_,_,_ in res for s in ss]
    folds=np.array([f for _,_,_,fs,_,_ in res for f in fs]); y=np.array([l for _,_,_,_,ys,_ in res for l in ys],dtype='float32')
    nf=sum(r[5] for r in res)
    for pth,_,_,_,_,_ in res:
        if os.path.exists(pth): os.remove(pth)
    if nf: print('  [%s] %d records skipped (malformed header)'%(tag,nf))
    return X,np.array(eids,dtype='U16'),np.array(srcs,dtype='U16'),folds,y
CACHE=os.path.join(CACHE_DIR,'m7_run2_data.npz')
if os.path.exists(CACHE):
    z=np.load(CACHE,allow_pickle=True)
    X18,eid18,src18,fold18,y18=z['X18'],z['eid18'],z['src18'],z['fold18'],z['y18']
    X9,eid9,y9=z['X9'],z['eid9'],z['y9']
    print('[guard] cache reloaded')
else:
    d18=meta[meta.fold.between(1,8)].reset_index(drop=True); d9=meta[meta.fold==9].reset_index(drop=True)
    print('extracting folds1-8 %d (%d WPW) + fold9 %d (%d WPW) @ %d ...'%(
        len(d18),int((d18.label==1).sum()),len(d9),int((d9.label==1).sum()),RESO))
    X18,eid18,src18,fold18,y18=_extract(d18,'f18'); X9,eid9,_,_,y9=_extract(d9,'f9')
    np.savez(CACHE,X18=X18,eid18=eid18,src18=src18,fold18=fold18,y18=y18,X9=X9,eid9=eid9,y9=y9)
    print('[guard] cache written')
print('folds1-8 X %s (%d WPW) | fold9 X %s (%d WPW)'%(X18.shape,int(y18.sum()),X9.shape,int(y9.sum())))
print('Block 2.1 OK.')


### Block 2.2: 1D-ResNet (identical to Run 1)

In [ ]:
# 1D-ResNet (modest) - identical to Run 1 (same architecture, frozen config).
import torch, torch.nn as nn
torch.set_num_threads(N_JOBS)
class BasicBlock1d(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.conv1=nn.Conv1d(cin,cout,7,stride=stride,padding=3,bias=False); self.bn1=nn.BatchNorm1d(cout)
        self.conv2=nn.Conv1d(cout,cout,7,padding=3,bias=False); self.bn2=nn.BatchNorm1d(cout)
        self.relu=nn.ReLU(inplace=True); self.down=None
        if stride!=1 or cin!=cout:
            self.down=nn.Sequential(nn.Conv1d(cin,cout,1,stride=stride,bias=False),nn.BatchNorm1d(cout))
    def forward(self,x):
        idt=x if self.down is None else self.down(x)
        o=self.relu(self.bn1(self.conv1(x))); o=self.bn2(self.conv2(o)); return self.relu(o+idt)
class ResNet1d(nn.Module):
    def __init__(self,ch=(16,32,64),pdrop=0.3):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv1d(12,ch[0],15,stride=2,padding=7,bias=False),
                                nn.BatchNorm1d(ch[0]),nn.ReLU(inplace=True),nn.MaxPool1d(4))
        self.layer1=BasicBlock1d(ch[0],ch[0],1)
        self.layer2=BasicBlock1d(ch[0],ch[1],2)
        self.layer3=BasicBlock1d(ch[1],ch[2],2)
        self.pool=nn.AdaptiveMaxPool1d(1)
        self.head=nn.Sequential(nn.Flatten(),nn.Dropout(pdrop),nn.Linear(ch[2],1))
    def forward(self,x):
        return self.head(self.pool(self.layer3(self.layer2(self.layer1(self.stem(x)))))).squeeze(1)
print('Block 2.2 OK | params:', sum(p.numel() for p in ResNet1d().parameters()))


### Block 2.3: Focal loss, augmentation, training (identical to Run 1)

In [ ]:
# Focal loss + augmentation + balanced-batch training; early-stop on val LOSS. Identical to Run 1.
# predict() casts float16 cache slices to float32 for the model.
def focal_loss(logits,targets,gamma=FOCAL_GAMMA,alpha=FOCAL_ALPHA):
    ce=nn.functional.binary_cross_entropy_with_logits(logits,targets,reduction='none')
    p=torch.sigmoid(logits); pt=torch.where(targets==1,p,1-p)
    a=torch.where(targets==1,torch.full_like(targets,alpha),torch.full_like(targets,1-alpha))
    return (a*(1-pt).pow(gamma)*ce).mean()
def augment(xb):
    T=xb.shape[2]; mx=max(1,int(0.04*T))
    xb=torch.roll(xb,int(torch.randint(-mx,mx+1,(1,)).item()),dims=2)
    xb=xb*(0.8+0.4*torch.rand(xb.shape[0],1,1)); xb=xb+0.02*torch.randn_like(xb)
    t=torch.linspace(0,1,T).view(1,1,-1)
    fb=0.5+2.0*torch.rand(xb.shape[0],1,1); ph=2*np.pi*torch.rand(xb.shape[0],1,1)
    xb=xb+0.05*torch.sin(2*np.pi*fb*t+ph)
    if torch.rand(1).item()<0.3: xb[:,int(torch.randint(0,12,(1,)).item()),:]=0.0
    return xb
def predict(model,X):
    model.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(X),256):
            out.append(torch.sigmoid(model(torch.tensor(np.ascontiguousarray(X[i:i+256],dtype=np.float32)))).numpy())
    return np.nan_to_num(np.concatenate(out))
def train_one(seed,Xtr,ytr,Xval,yval):
    torch.manual_seed(seed); np.random.seed(seed); rng=np.random.default_rng(seed)
    pos=np.where(ytr==1)[0]; neg=np.where(ytr==0)[0]; half=BATCH//2
    Xv=torch.tensor(np.ascontiguousarray(Xval,dtype=np.float32)); yv=torch.tensor(yval)
    model=ResNet1d(); opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WD)
    steps=max(1,len(ytr)//BATCH); best=1e9; best_state=None; bad=0; last=0
    for ep in range(EPOCHS):
        last=ep+1; model.train()
        for _ in range(steps):
            bi=np.concatenate([rng.choice(pos,half,True),rng.choice(neg,half,True)])
            xb=augment(torch.tensor(np.ascontiguousarray(Xtr[bi],dtype=np.float32))); yb=torch.tensor(ytr[bi])
            opt.zero_grad(); loss=focal_loss(model(xb),yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl=[focal_loss(model(Xv[i:i+256]),yv[i:i+256]).item() for i in range(0,len(Xv),256)]
        vloss=float(np.mean(vl))
        if vloss<best-1e-4: best=vloss; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=PATIENCE: break
    model.load_state_dict(best_state); return model,last
print('Block 2.3 OK.')


### Block 2.4: OOF folds 1-8 (8 folds x 5 seeds) + fold9 by model reuse

In [ ]:
# OOF folds 1-8 with PER-FOLD and PER-SEED checkpoints (resume-safe; interruption costs <= 1 seed ~17 min).
# Per held-out fold f: train on the other 7 folds (all WPW + subsampled negs), seeded stratified internal val
# for early-stop, 5-seed ensemble; predict held-out fold (-> OOF, true prevalence) and fold9 (-> reuse).
# Shared accumulators (f9_acc, f9_n) updated ONLY at fold completion -> resume never double-counts.
CKPT=os.path.join(CACHE_DIR,'m7_run2_oof_ckpt.npz')
if os.path.exists(CKPT):
    ck=np.load(CKPT,allow_pickle=True)
    oof=ck['oof'].astype('float32'); f9_acc=ck['f9_acc'].astype('float64'); f9_n=int(ck['f9_n'])
    train_ap_folds=list(ck['train_ap_folds']); done_folds=set(int(x) for x in ck['done_folds'])
    print('[ckpt] resumed | folds done %s'%sorted(done_folds))
else:
    oof=np.full(len(y18),np.nan,dtype='float32'); f9_acc=np.zeros(len(y9),dtype='float64'); f9_n=0
    train_ap_folds=[]; done_folds=set()
t_all=time.time()
for f in range(1,9):
    if f in done_folds:
        print('  fold %d already done (ckpt) -> skip'%f); continue
    te_idx=np.where(fold18==f)[0]; pool_idx=np.where(fold18!=f)[0]
    rng=np.random.default_rng(1000+f)
    pos=pool_idx[y18[pool_idx]==1]; negall=pool_idx[y18[pool_idx]==0]
    neg=rng.choice(negall,min(N_NEG_TRAIN,len(negall)),replace=False)
    vp=pos.copy(); vn=neg.copy(); rng.shuffle(vp); rng.shuffle(vn)
    nvp=max(6,int(VAL_FRAC*len(vp))); nvn=int(VAL_FRAC*len(vn))
    val_idx=np.concatenate([vp[:nvp],vn[:nvn]]); pool=np.concatenate([pos,neg]); tr_idx=np.setdiff1d(pool,val_idx)
    Xtr=np.ascontiguousarray(X18[tr_idx],dtype=np.float32); ytr=y18[tr_idx]
    Xva=X18[val_idx]; yva=y18[val_idx]; Xte=X18[te_idx]
    sck=os.path.join(CACHE_DIR,'m7_run2_fold%d_seeds.npz'%f)                            # per-seed checkpoint
    if os.path.exists(sck):
        sc=np.load(sck); seed_te=[r for r in sc['te']]; seed_tr=[r for r in sc['tr']]
        f9_local=sc['f9'].astype('float64'); done_s=int(sc['done'])
        print('  fold %d: resume from seed %d/%d'%(f,done_s,len(SEEDS)))
    else:
        seed_te=[]; seed_tr=[]; f9_local=np.zeros(len(y9),dtype='float64'); done_s=0
    for si,s in enumerate(SEEDS):
        if si<done_s: continue
        model,ne=train_one(s,Xtr,ytr,Xva,yva)
        seed_te.append(predict(model,Xte)); seed_tr.append(predict(model,Xtr)); f9_local=f9_local+predict(model,X9)
        np.savez(sck,te=np.array(seed_te),tr=np.array(seed_tr),f9=f9_local,done=si+1)
    oof[te_idx]=np.mean(seed_te,0); f9_acc=f9_acc+f9_local; f9_n+=len(SEEDS)
    train_ap_folds.append(float(average_precision_score(ytr,np.mean(seed_tr,0)))); done_folds.add(f)
    np.savez(CKPT,oof=oof,f9_acc=f9_acc,f9_n=f9_n,train_ap_folds=np.array(train_ap_folds),
             done_folds=np.array(sorted(done_folds)))                                   # per-fold checkpoint
    if os.path.exists(sck): os.remove(sck)
    done=~np.isnan(oof)
    print('  fold %d done | held-out %d (%d WPW) | OOF-AP so far %.3f | %.1f min'%(
        f,len(te_idx),int(y18[te_idx].sum()),average_precision_score(y18[done],oof[done]),(time.time()-t_all)/60))
    del Xtr,Xva,Xte
f9=(f9_acc/max(f9_n,1)).astype('float32')
np.save(os.path.join(MODELS,'m7_oof_folds1_8.npy'),oof); np.save(os.path.join(MODELS,'m7_fold9_scores.npy'),f9)
pd.DataFrame({'ecg_id':eid18,'source':src18,'fold':fold18,'label':y18,'m7_oof':oof}).to_csv(
    os.path.join(PROCESSED,'m7_combined_oof.csv'),index=False)
print('Block 2.4 OK | total %.1f min | models trained %d'%((time.time()-t_all)/60,f9_n))

### Block 2.5: Honest OOF metric + per-source AP + standardized fold9 eval

In [ ]:
# Honest OOF metric (folds 1-8) = the number comparable to M3/M4. Plus per-source AP + standardized fold9 eval.
ok=~np.isnan(oof)
OOF_AP=float(average_precision_score(y18[ok],oof[ok])); OOF_AUC=float(roc_auc_score(y18[ok],oof[ok]))
ap_lo,ap_hi=_boot_ci(y18[ok].astype(int),oof[ok],average_precision_score)
ap_train=float(np.mean(train_ap_folds)); gap=ap_train-OOF_AP
print('=== M7 Run 2 OOF (folds 1-8, %d WPW) ==='%int(y18[ok].sum()))
print('  OOF AP  %.3f CI95[%.3f,%.3f]   REF: M3 %.3f | M4 %.3f | vote %.3f'%(OOF_AP,ap_lo,ap_hi,REF['M3'],REF['M4'],REF['vote_M3_M4']))
print('  OOF AUC %.3f | gap train-OOF %+.3f (train %.3f)'%(OOF_AUC,gap,ap_train))
for s in ['ptb','ningbo']:
    m=ok & np.array([str(x).lower().startswith(s) for x in src18])
    if m.sum() and y18[m].sum()>=2:
        print('   OOF AP [%-6s] %.3f (%d WPW / %d ECG)'%(s,float(average_precision_score(y18[m],oof[m])),int(y18[m].sum()),int(m.sum())))
evaluate_standard(name='M7run2_oof_5000',y_oof=y18[ok],score_oof=oof[ok],y_test=y9,score_test=f9,
                  figures_dir=FIG,metrics_dir=METRICS,ap_train=ap_train,
                  extra={'run':'m7_run2','resolution':RESO,'OOF_AP':OOF_AP,'OOF_AP_CI':[ap_lo,ap_hi],
                         'OOF_AUC':OOF_AUC,'gap_train_oof':gap})
print('Block 2.5 OK.')


### Block 2.6: Read-only ensemble preview (rho, rank-vote) aligned by ecg_id

In [ ]:
# READ-ONLY ensemble preview (informational for C1; NO tuning). Align by (ecg_id, source) with M3/M4 OOF.
m3=pd.read_csv(os.path.join(PROCESSED,'m3_combined_oof.csv'),dtype={'ecg_id':str})[['ecg_id','source','proba_raw']].rename(columns={'proba_raw':'m3'})
m4=pd.read_csv(os.path.join(PROCESSED,'m4_combined_oof.csv'),dtype={'ecg_id':str})[['ecg_id','source','proba_raw']].rename(columns={'proba_raw':'m4'})
m7=pd.DataFrame({'ecg_id':np.asarray(eid18,dtype=str),'source':np.asarray(src18,dtype=str),'m7':oof,'label':y18})
m7=m7[~np.isnan(m7.m7)]
D=m7.merge(m3,on=['ecg_id','source']).merge(m4,on=['ecg_id','source'])
print('aligned (M3 ∩ M4 ∩ M7): %d rows, %d WPW'%(len(D),int(D.label.sum())))
def rk(a): return pd.Series(np.asarray(a)).rank(pct=True).values
rho_m3=float(spearmanr(D.m7,D.m3).correlation); rho_m4=float(spearmanr(D.m7,D.m4).correlation)
vote2=rk(D.m3)+rk(D.m4); vote3=rk(D.m3)+rk(D.m4)+rk(D.m7)
ap2=float(average_precision_score(D.label,vote2)); ap3=float(average_precision_score(D.label,vote3))
rho_vote=float(spearmanr(D.m7,vote2).correlation)
print('  rho(M7,M3) %.3f | rho(M7,M4) %.3f | rho(M7, vote M3+M4) %.3f'%(rho_m3,rho_m4,rho_vote))
print('  rank-vote AP : M3+M4 %.3f -> M3+M4+M7 %.3f   (delta %+.3f)'%(ap2,ap3,ap3-ap2))
json.dump({'n_aligned':int(len(D)),'OOF_AP':OOF_AP,'rho_M3':rho_m3,'rho_M4':rho_m4,'rho_vote':rho_vote,
           'rankvote_M3M4':ap2,'rankvote_M3M4M7':ap3,'delta':ap3-ap2},
          open(os.path.join(METRICS,'m7_run2_ensemble_preview.json'),'w'),indent=2)
print('Block 2.6 OK.')


### Block 2.7: Gate C1 (pre-registered)

In [ ]:
# Gate C1 (pre-registered): serious ensemble candidate IFF OOF >= 0.65 AND max rho < 0.50.
# rho<0.5 is NECESSARY not SUFFICIENT. Any override of a FAIL must be journaled.
rho_max=float(max(rho_m3,rho_m4))
pass_oof=bool(OOF_AP>=C1_OOF); pass_rho=bool(rho_max<C1_RHO); C1=bool(pass_oof and pass_rho)
print('=== GATE C1 ===')
print('  OOF AP %.3f   (>= %.2f ? %s)'%(OOF_AP,C1_OOF,pass_oof))
print('  max rho(M7,{M3,M4}) %.3f   (< %.2f ? %s)'%(rho_max,C1_RHO,pass_rho))
print('  C1: %s'%('PASS -> pretraining Runs 8-10 UNLOCKED' if C1 else 'FAIL -> pretraining Runs 8-10 NOT opened (default policy)'))
print('  rho<0.5 NECESSARY not SUFFICIENT (M5 had rho~0.19 and did not help).')
print('  Overriding a FAIL must be journaled with the real OOF, rho, and remaining time.')
print('  Next: Runs 3-6 (A/B: imbalance -> augmentation -> regularization -> architecture LAST) run regardless;')
print('        the C1 verdict only gates the pretraining block (Runs 8-10).')
json.dump({'OOF_AP':OOF_AP,'rho_max':rho_max,'pass_oof':pass_oof,'pass_rho':pass_rho,'C1_pass':C1},
          open(os.path.join(METRICS,'m7_run2_C1.json'),'w'),indent=2)
print('Block 2.7 OK.')
